# Fig 8 — Convergence to the exact NESS (empirical error bars)

Dynamic Rodeo circuit on a noiseless `AerSimulator`. Error bars are the **empirical run-to-run std** over `N_REPEATS` independent, well-separated seeds.

The **canonical** figure (more reps / 40k shots) is produced by `reproduce/circuits/standalone_aer_validation_empirical.py` — run it locally (~30-40 min).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../.."))
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rcParams
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from rodeo_ness import circuits as C
rcParams.update({"font.family":"serif","font.serif":["DejaVu Serif"],
                 "mathtext.fontset":"dejavuserif","axes.linewidth":0.8})
BLUE,RED,GREEN = "#0072B2","#d62728","#009E73"
# NOTE: small N_REPEATS/SHOTS here so the notebook runs quickly. The CANONICAL
# figure (10-20 reps x 40k shots) is produced by
# reproduce/circuits/standalone_aer_validation_empirical.py -- run it locally.
N_REPEATS, SHOTS = 4, 15000
recs = C.run_convergence_sweep(h=0.5, n_values=range(1,13), shots=SHOTS,
                               seed=0, n_repeats=N_REPEATS)
exact = C.exact_sz(0.5)
n   = np.array([r["n"] for r in recs], float)
sz  = np.array([r["sz_estimate"] for r in recs], float)
sd  = np.array([r["sz_err"] for r in recs], float)
ae  = np.array([r["abs_error_sz"] for r in recs], float)
avg = np.array([r["avg_executed_cycles"] for r in recs], float)
avgs= np.array([r["avg_executed_err"] for r in recs], float)
sav = np.array([r["cycle_saving"] for r in recs], float)
savs= np.array([r["cycle_saving_err"] for r in recs], float)

In [ ]:
fig,(axA,axB)=plt.subplots(1,2,figsize=(11,4.5))
axA.errorbar(n,sz,yerr=sd,fmt="o-",color=BLUE,ms=6,lw=1.5,capsize=3,mfc="white",mec=BLUE,ecolor=BLUE,label="Aer (mean $\\pm$ std)")
axA.axhline(exact,ls="--",color=RED,lw=1.5,label=r"exact NESS $-1/3$")
axA.set_xlabel("number of Rodeo cycles $n$"); axA.set_ylabel(r"$\langle\hat\sigma_z\rangle$")
axA.set_title(f"(a) convergence to the NESS ({N_REPEATS} reps, {SHOTS//1000}k shots)")
axA.legend(fontsize=9,loc="lower right"); axA.grid(alpha=0.2)
axin=inset_axes(axA,width="52%",height="42%",loc="upper right",borderpad=1.1); m=n>=6
axin.errorbar(n[m],sz[m],yerr=sd[m],fmt="o-",color=BLUE,ms=4,lw=1.2,capsize=2,mfc="white",mec=BLUE,ecolor=BLUE)
axin.axhline(exact,ls="--",color=RED,lw=1.2); axin.set_ylim(exact-0.08,exact+0.08)
axin.set_title(r"zoom: $n\geq6$",fontsize=8); axin.tick_params(labelsize=7); axin.grid(alpha=0.2)
axB.semilogy(n,ae,"o-",color=BLUE,ms=6,lw=1.5); floor=float(np.nanmedian(sd[n>=6]))
axB.axhspan(1e-4,floor,color="0.85",zorder=0)
axB.axhline(floor,ls=":",color="0.4",lw=1.2,label=f"empirical shot-noise floor $\\approx${floor:.3f}")
axB.set_xlabel("number of Rodeo cycles $n$"); axB.set_ylabel(r"$|\langle\hat\sigma_z\rangle-(-1/3)|$")
axB.set_title("(b) absolute error falls to the shot-noise floor"); axB.legend(fontsize=9,loc="upper right"); axB.grid(alpha=0.2,which="both")
fig.tight_layout(); fig.savefig("aer_convergence.pdf",bbox_inches="tight"); plt.show()